# Lineup Quality Walk-Forward — 2025 Season

Does lower MAE late in the season actually produce better DFS lineups?

**Protocol:**
- Train 3 models at different points of season maturity
- Run the full predict → optimize pipeline for each model over its corresponding test window
- Compare win rate and average lineup FPTS across the three slices

| Slice | Train data | Test window |
|---|---|---|
| Pre-season | 2021–2024 only | May + Jun 2025 |
| Mid-season | + May + Jun 2025 | Jul + Aug 2025 |
| Late-season | + Jul + Aug 2025 | Sep 2025 |

> ⚠️ Note: September has only ~538 player rows — results for that slice have higher variance.

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src import config
from src.inference.bulk_backtest import run_bulk_backtest

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
HOLDOUT_SEASON = 2025
N_ESTIMATORS   = 121

XGB_PARAMS = {
    **config.XGB_PARAMS,
    'n_estimators': N_ESTIMATORS,
    'device': 'cpu',   # CPU avoids libnvrtc dependency in notebooks
}

SLICES = [
    {
        'label':      'Pre-season',
        'train_end':  None,           # 2021-2024 only
        'test_start': '2025-05-01',
        'test_end':   '2025-06-30',
    },
    {
        'label':      'Mid-season',
        'train_end':  '2025-06-30',   # 2021-2024 + May/Jun 2025
        'test_start': '2025-07-01',
        'test_end':   '2025-08-31',
    },
    {
        'label':      'Late-season',
        'train_end':  '2025-08-31',   # 2021-2024 + May-Aug 2025
        'test_start': '2025-09-01',
        'test_end':   '2025-09-30',
    },
]

In [ ]:
# ── Load & split data ─────────────────────────────────────────────────────────
df = pd.read_csv(config.PROCESSED_DATA_DIR / 'training_features.csv')
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])

to_drop      = config.DROPPED_FEATURES + [config.TARGET_COL]
actual_drops = [c for c in to_drop if c in df.columns]
feature_cols = df.drop(columns=actual_drops).select_dtypes(include=['number']).columns.tolist()

base_df   = df[df['SEASON'] != HOLDOUT_SEASON].copy()
season_df = df[df['SEASON'] == HOLDOUT_SEASON].copy()

print(f"Base rows (2021-2024): {len(base_df):,}")
print(f"2025 rows            : {len(season_df):,}")
print(f"Feature columns      : {len(feature_cols)}")

In [ ]:
# ── Train each slice model and run backtest ───────────────────────────────────
results        = []
random_results = []

for s in SLICES:
    print("\n" + "#"*60)
    print(f"# SLICE: {s['label']}")
    print("#"*60)

    # Build training set
    if s['train_end'] is None:
        train_df = base_df
        print(f"Training on: 2021–2024 only ({len(train_df):,} rows)")
    else:
        in_season_train = season_df[season_df['GAME_DATE'] <= s['train_end']]
        train_df = pd.concat([base_df, in_season_train])
        print(f"Training on: 2021–2024 + 2025 through {s['train_end']} ({len(train_df):,} rows)")

    # Train model
    X_train = train_df[feature_cols].astype('float64').values
    y_train = train_df[config.TARGET_COL].values

    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(X_train, y_train, verbose=False)
    print(f"Model trained.")

    # --- Model backtest ---
    summary = run_bulk_backtest(
        start_date=s['test_start'],
        end_date=s['test_end'],
        model=model,
    )
    if summary:
        summary['label'] = s['label']
        results.append(summary)

    # --- Random baseline (same window, no model) ---
    rand_summary = run_bulk_backtest(
        start_date=s['test_start'],
        end_date=s['test_end'],
        random_baseline=True,
    )
    if rand_summary:
        rand_summary['label'] = s['label']
        random_results.append(rand_summary)

print("\n✅ All slices complete.")

In [ ]:
# ── Results table ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).set_index('label')
random_df  = pd.DataFrame(random_results).set_index('label')

print(f"{'Slice':<15} | {'Model Win%':>10} | {'Random Win%':>11} | {'Model FPTS':>10} | {'Random FPTS':>11}")
print("-" * 68)
for label in results_df.index:
    m = results_df.loc[label]
    r = random_df.loc[label] if label in random_df.index else None
    rwin = f"{r['win_rate']:>10.1f}%" if r is not None else "        N/A"
    rfpts = f"{r['avg_fpts']:>11.2f}" if r is not None else "        N/A"
    print(f"{label:<15} | {m['win_rate']:>9.1f}% | {rwin} | {m['avg_fpts']:>10.2f} | {rfpts}")

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
labels = results_df.index.tolist()
x      = np.arange(len(labels))
width  = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

model_win  = results_df['win_rate'].tolist()
random_win = [random_df.loc[l, 'win_rate'] if l in random_df.index else 0 for l in labels]
model_fpts  = results_df['avg_fpts'].tolist()
random_fpts = [random_df.loc[l, 'avg_fpts'] if l in random_df.index else 0 for l in labels]

# Left: Win Rate
b1 = ax1.bar(x - width/2, model_win,  width, label='Model',  color='steelblue')
b2 = ax1.bar(x + width/2, random_win, width, label='Random', color='darkorange')
ax1.axhline(50, color='black', linestyle='--', linewidth=1, label='50% breakeven')
ax1.set_title('Win Rate: Model vs Random', fontsize=13)
ax1.set_ylabel('Win Rate (%)')
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_ylim(0, 110)
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.legend()
for bar, val in zip(list(b1) + list(b2), model_win + random_win):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.0f}%', ha='center', va='bottom', fontsize=9)

# Right: Avg FPTS
b3 = ax2.bar(x - width/2, model_fpts,  width, label='Model',  color='steelblue')
b4 = ax2.bar(x + width/2, random_fpts, width, label='Random', color='darkorange')
ax2.axhline(150, color='tomato', linestyle='--', linewidth=1, label='Cash line (150)')
ax2.set_title('Avg Lineup FPTS: Model vs Random', fontsize=13)
ax2.set_ylabel('Fantasy Points')
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.legend()
for bar, val in zip(list(b3) + list(b4), model_fpts + random_fpts):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Does more in-season data produce better lineups?', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('lineup_quality_2025.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: lineup_quality_2025.png')